# train-eval-mode-branch — worked example 1: LayerNorm behaves identically in train and eval mode

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `train-eval-mode-branch`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

Unlike `BatchNorm`, `LayerNorm` normalizes over the feature dimension of each individual sample rather than across the batch. This means it does not maintain running statistics, so its output is identical in `model.train()` and `model.eval()`. Understanding which modules are mode-sensitive is key to writing correct inference code.

## Worked solution

**Step 1 – Build a LayerNorm.** `nn.LayerNorm(features)` normalizes the last dimension of each input independently. It has learnable `weight` and `bias` parameters but no running mean/variance.

**Step 2 – Forward in train mode.** We call `ln.train()` and compute `out_train = ln(x)`. Because LayerNorm uses per-sample statistics (not batch statistics), the output depends only on the current input, not on accumulated stats.

**Step 3 – Forward in eval mode.** We call `ln.eval()` and compute `out_eval = ln(x)`. Since there are no running statistics to freeze, the result is identical to the training-mode output for the same input.

**Step 4 – Compare.** We check `t.allclose(out_train, out_eval)`. The result is True, which contrasts with `BatchNorm` (where the two outputs differ because eval mode uses frozen running stats instead of batch stats).

**Key insight:** The `model.training` flag matters for modules that branch on it — `Dropout` drops units, `BatchNorm` switches between batch stats and running stats. Pure compute modules like `LayerNorm` are unaffected.

In [ ]:
import torch as t
import torch.nn as nn

def worked1_layernorm_mode_invariant():
    """
    LayerNorm output is identical in train and eval mode.
    Returns dict with outputs and allclose result.
    """
    t.manual_seed(11)
    features = 6
    ln = nn.LayerNorm(features)
    x = t.randn(4, features)

    ln.train()
    out_train = ln(x)
    flag_train = ln.training   # True

    ln.eval()
    out_eval = ln(x)
    flag_eval = ln.training    # False

    are_equal = t.allclose(out_train, out_eval)
    return {
        'out_train': out_train,
        'out_eval': out_eval,
        'flag_train': flag_train,
        'flag_eval': flag_eval,
        'identical': are_equal,
    }

result = worked1_layernorm_mode_invariant()
print('train flag:', result['flag_train'])
print('eval flag:', result['flag_eval'])
print('outputs identical:', result['identical'])